In [ ]:

import pandas as pd
import numpy as np

# Reproducible simulated dataset
np.random.seed(42)
n = 5000

# Bus routes and route distances
routes = {
    "R01": ("Central - Airport", 22),
    "R02": ("Central - Railway Station", 12),
    "R03": ("Central - IT Park", 28),
    "R04": ("Market - University", 15),
    "R05": ("Central - Hospital", 10),
    "R06": ("Airport - IT Park", 18),
    "R07": ("University - Market", 14),
    "R08": ("Central - Industrial Estate", 30),
    "R09": ("Railway Station - IT Park", 25),
    "R10": ("Central - Beach", 16),
    "R11": ("Market - Airport", 24),
    "R12": ("University - Hospital", 9)
}

route_ids = list(routes.keys())
route_id = np.random.choice(route_ids, n)
route_names = [routes[r][0] for r in route_id]
distance = np.array([routes[r][1] for r in route_id])

# Dates across January and February 2026
trip_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2026-01-01", "2026-02-28"),
        n
    )
)

# More trips during morning and evening peak hours
hour_probabilities = np.array([
    2, 3, 5, 8, 10, 8, 5, 4, 4,
    4, 4, 5, 7, 10, 9, 5, 3, 2
], dtype=float)

# Normalize probabilities so their total becomes 1
hour_probabilities = hour_probabilities / hour_probabilities.sum()

hours = np.random.choice(
    np.arange(5, 23),
    n,
    p=hour_probabilities
)

minutes = np.random.choice([0, 15, 30, 45], n)

scheduled = (
    trip_dates
    + pd.to_timedelta(hours, unit="h")
    + pd.to_timedelta(minutes, unit="m")
)

# Bus and passenger details
bus_id = np.random.choice(
    [f"B{i:03d}" for i in range(1, 61)], n
)

bus_capacity = np.random.choice([40, 50, 60], n)
passenger_count = np.random.randint(5, 61, n)
passenger_count = np.minimum(passenger_count, bus_capacity)

# Simulated delays and trip status
delay_minutes = np.maximum(
    0, np.round(np.random.normal(8, 9, n))
).astype(int)

cancelled = np.random.random(n) < 0.03
delay_minutes[cancelled] = 0

trip_status = np.where(
    cancelled, "Cancelled",
    np.where(delay_minutes > 10, "Delayed", "On Time")
)

passenger_count[cancelled] = 0

# Revenue and fuel
ticket_fare = np.random.randint(10, 61, n)
ticket_revenue = passenger_count * ticket_fare

fuel_litres = np.round(
    distance * np.random.uniform(0.20, 0.35, n), 2
)
fuel_litres[cancelled] = 0

fuel_cost = np.round(fuel_litres * 95, 2)

# Create the final dataset
df = pd.DataFrame({
    "Trip_ID": [f"T{i:05d}" for i in range(1, n + 1)],
    "Trip_Date": trip_dates.strftime("%Y-%m-%d"),
    "Route_ID": route_id,
    "Route_Name": route_names,
    "Bus_ID": bus_id,
    "Scheduled_Departure": scheduled.strftime("%Y-%m-%d %H:%M:%S"),
    "Actual_Departure": (
        scheduled + pd.to_timedelta(delay_minutes, unit="m")
    ).strftime("%Y-%m-%d %H:%M:%S"),
    "Distance_KM": distance,
    "Passenger_Count": passenger_count,
    "Bus_Capacity": bus_capacity,
    "Occupancy_Pct": np.round(
        passenger_count / bus_capacity * 100, 2
    ),
    "Delay_Minutes": delay_minutes,
    "Trip_Status": trip_status,
    "Ticket_Fare_INR": ticket_fare,
    "Ticket_Revenue_INR": ticket_revenue,
    "Fuel_Consumed_Litres": fuel_litres,
    "Fuel_Cost_INR": fuel_cost
})

# Save dataset
df.to_csv("Urban_Bus_Operations.csv", index=False)

print("Dataset created successfully!")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print(df.head())

Dataset created successfully!
Total rows: 5000
Total columns: 17
  Trip_ID   Trip_Date Route_ID                   Route_Name Bus_ID  \
0  T00001  2026-01-15      R07          University - Market   B002   
1  T00002  2026-01-20      R04          Market - University   B034   
2  T00003  2026-01-29      R11             Market - Airport   B027   
3  T00004  2026-02-28      R08  Central - Industrial Estate   B060   
4  T00005  2026-02-11      R05           Central - Hospital   B054   

   Scheduled_Departure     Actual_Departure  Distance_KM  Passenger_Count  \
0  2026-01-15 17:15:00  2026-01-15 17:15:00           14                8   
1  2026-01-20 18:30:00  2026-01-20 18:30:00           15               39   
2  2026-01-29 16:30:00  2026-01-29 16:33:00           24               45   
3  2026-02-28 09:30:00  2026-02-28 09:33:00           30               26   
4  2026-02-11 06:15:00  2026-02-11 06:15:00           10               40   

   Bus_Capacity  Occupancy_Pct  Delay_Minutes Trip_

In [ ]:
from google.colab import files
files.download("Urban_Bus_Operations.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Check duplicate rows
print("Total Duplicate Rows:", df.duplicated().sum())

# Check duplicate Trip IDs
print("Duplicate Trip IDs:", df["Trip_ID"].duplicated().sum())

Total Duplicate Rows: 0
Duplicate Trip IDs: 0


In [ ]:
import pandas as pd

df = pd.read_csv("Urban_Bus_Operations.csv")

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (5000, 17)

Column Names:
['Trip_ID', 'Trip_Date', 'Route_ID', 'Route_Name', 'Bus_ID', 'Scheduled_Departure', 'Actual_Departure', 'Distance_KM', 'Passenger_Count', 'Bus_Capacity', 'Occupancy_Pct', 'Delay_Minutes', 'Trip_Status', 'Ticket_Fare_INR', 'Ticket_Revenue_INR', 'Fuel_Consumed_Litres', 'Fuel_Cost_INR']

First 5 Rows:


,Trip_ID,Trip_Date,Route_ID,Route_Name,Bus_ID,Scheduled_Departure,Actual_Departure,Distance_KM,Passenger_Count,Bus_Capacity,Occupancy_Pct,Delay_Minutes,Trip_Status,Ticket_Fare_INR,Ticket_Revenue_INR,Fuel_Consumed_Litres,Fuel_Cost_INR
0,T00001,2026-01-15,R07,University - Market,B002,2026-01-15 17:15:00,2026-01-15 17:15:00,14,8,60,13.33,0,On Time,40,320,4.25,403.75
1,T00002,2026-01-20,R04,Market - University,B034,2026-01-20 18:30:00,2026-01-20 18:30:00,15,39,50,78.00,0,On Time,57,2223,3.09,293.55
2,T00003,2026-01-29,R11,Market - Airport,B027,2026-01-29 16:30:00,2026-01-29 16:33:00,24,45,50,90.00,3,On Time,18,810,8.00,760.00
3,T00004,2026-02-28,R08,Central - Industrial Estate,B060,2026-02-28 09:30:00,2026-02-28 09:33:00,30,26,60,43.33,3,On Time,21,546,8.97,852.15
4,T00005,2026-02-11,R05,Central - Hospital,B054,2026-02-11 06:15:00,2026-02-11 06:15:00,10,40,40,100.00,0,On Time,53,2120,2.41,228.95



Missing Values:
Trip_ID                 0
Trip_Date               0
Route_ID                0
Route_Name              0
Bus_ID                  0
Scheduled_Departure     0
Actual_Departure        0
Distance_KM             0
Passenger_Count         0
Bus_Capacity            0
Occupancy_Pct           0
Delay_Minutes           0
Trip_Status             0
Ticket_Fare_INR         0
Ticket_Revenue_INR      0
Fuel_Consumed_Litres    0
Fuel_Cost_INR           0
dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Trip_ID               5000 non-null   object 
 1   Trip_Date             5000 non-null   object 
 2   Route_ID              5000 non-null   object 
 3   Route_Name            5000 non-null   object 
 4   Bus_ID                5000 non-null   object 
 5   Scheduled_Departure   5000 non-null   object 
 6   Actual_Departure      5000 non-null   object 
 7   Distance_KM           5000 non-null   int64  
 8   Passenger_Count       5000 non-null   int64  
 9   Bus_Capacity          5000 non-null   int64  
 10  Occupancy_Pct         5000 non-null   float64
 11  Delay_Minutes         5000 non-null   int64  
 12  Trip_Status           5000 non-null   object 
 13  Ticket_Fare_INR       5000 non-null   int64  
 14  Ticket_Revenue_INR    5000 non-null   int64  
 15  Fuel_Consumed_Litres 

In [ ]:
# Convert date and time columns
df["Trip_Date"] = pd.to_datetime(df["Trip_Date"], errors="coerce")

df["Scheduled_Departure"] = pd.to_datetime(
    df["Scheduled_Departure"], errors="coerce"
)

df["Actual_Departure"] = pd.to_datetime(
    df["Actual_Departure"], errors="coerce"
)

# Verify the data types
print(df[[
    "Trip_Date",
    "Scheduled_Departure",
    "Actual_Departure"
]].dtypes)

# Check for invalid dates
print("\nMissing values after conversion:")
print(df[[
    "Trip_Date",
    "Scheduled_Departure",
    "Actual_Departure"
]].isnull().sum())

Trip_Date              datetime64[ns]
Scheduled_Departure    datetime64[ns]
Actual_Departure       datetime64[ns]
dtype: object

Missing values after conversion:
Trip_Date              0
Scheduled_Departure    0
Actual_Departure       0
dtype: int64


In [ ]:
print("Minimum Delay:", df["Delay_Minutes"].min())
print("Maximum Delay:", df["Delay_Minutes"].max())
print("Negative Delays:", (df["Delay_Minutes"] < 0).sum())

print("\nTrip Status Counts:")
print(df["Trip_Status"].value_counts())

Minimum Delay: 0
Maximum Delay: 43
Negative Delays: 0

Trip Status Counts:
Trip_Status
On Time      2982
Delayed      1861
Cancelled     157
Name: count, dtype: int64


In [ ]:
print("Minimum Passengers:", df["Passenger_Count"].min())
print("Maximum Passengers:", df["Passenger_Count"].max())

print("Capacity Violations:",
      (df["Passenger_Count"] > df["Bus_Capacity"]).sum())

print("Revenue Mismatch:",
      (df["Ticket_Revenue_INR"] !=
       df["Passenger_Count"] * df["Ticket_Fare_INR"]).sum())

print("Negative Revenue:",
      (df["Ticket_Revenue_INR"] < 0).sum())

Minimum Passengers: 0
Maximum Passengers: 60
Capacity Violations: 0
Revenue Mismatch: 0
Negative Revenue: 0


In [ ]:
# Create a cleaned dataset
clean_df = df.drop_duplicates().copy()

# Reset row numbering
clean_df = clean_df.reset_index(drop=True)

# Save the cleaned dataset
clean_df.to_csv("Urban_Bus_Operations_Cleaned.csv", index=False)

print("Cleaned dataset created successfully!")
print("Total rows:", len(clean_df))
print("Total columns:", len(clean_df.columns))# Create a cleaned dataset
clean_df = df.drop_duplicates().copy()

# Reset row numbering
clean_df = clean_df.reset_index(drop=True)

# Save the cleaned dataset
clean_df.to_csv("Urban_Bus_Operations_Cleaned.csv", index=False)

print("Cleaned dataset created successfully!")
print("Total rows:", len(clean_df))
print("Total columns:", len(clean_df.columns))

Cleaned dataset created successfully!
Total rows: 5000
Total columns: 17
Cleaned dataset created successfully!
Total rows: 5000
Total columns: 17


In [ ]:
from google.colab import files
files.download("Urban_Bus_Operations_Cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(df.columns.tolist())
print("Duplicate column names:", df.columns[df.columns.duplicated()].tolist())

['Trip_ID', 'Trip_Date', 'Route_ID', 'Route_Name', 'Bus_ID', 'Scheduled_Departure', 'Actual_Departure', 'Distance_KM', 'Passenger_Count', 'Bus_Capacity', 'Occupancy_Pct', 'Delay_Minutes', 'Trip_Status', 'Ticket_Fare_INR', 'Ticket_Revenue_INR', 'Fuel_Consumed_Litres', 'Fuel_Cost_INR']
Duplicate column names: []
